# SRP-conditions-predict-using-ML - Kaggle runner

**This notebook is deliberately thin. It contains NO experiment logic.**
Everything scientific lives in the repository, under version control. This notebook
only: clones, installs, points DATA_ROOT at the attached dataset, calls ONE script,
and copies `artifacts/` back out.

If you find yourself editing an experiment parameter here, stop: edit `configs/*.yaml`
in the repository, push, and re-run this notebook.

See `docs/KAGGLE_SETUP.md` for how to attach the dataset and set the accelerator.

---
**Session boundary and resume.** Kaggle sessions end after ~12 h. Every script is
resumable from `artifacts/registry.jsonl`, and there are two ways to get that file
back into a new session:

1. **git is the PRIMARY path.** `artifacts/registry.jsonl` is committed (it has an
   exception in `.gitignore`). At the end of every session you commit it; the next
   session's `git clone` in cell 1 restores it, and every script skips the runs it
   already contains. Nothing to upload, nothing to attach.
2. **`RESUME_FROM` is the documented FALLBACK** (cell 3), for when you cannot push
   -- no token to hand, a rejected push, a session that died before you could
   commit. It restores `artifacts/` from a Kaggle Dataset instead. You should not
   need it in the normal loop.

Either way, run the last cell before the session ends so you have the bundle to
commit from.


## 1. Clone the repository


In [ ]:
# A private repo needs a token in the URL or a Kaggle Secret.
REPO_URL = 'https://github.com/akiraraihaan/SRP-conditions-predict-using-ML.git'
BRANCH   = 'main'

import os, shutil, subprocess, sys
WORK = '/kaggle/working/SRP-conditions-predict-using-ML'

# Stand outside the tree before deleting it: this cell is re-run whenever the
# repository changes, and it ends inside WORK, so a second run would otherwise
# rmtree the process's own working directory and git clone would exit 128.
os.chdir('/kaggle/working')

if os.path.exists(WORK):
    shutil.rmtree(WORK)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,WORK], check=True)
os.chdir(WORK)
print('cwd:', os.getcwd())
subprocess.run(['git','log','-1','--oneline'], check=False)

# The clone is what restores the resume state: registry.jsonl is committed, so
# everything already finished in an earlier session arrives with it.
import os.path
reg = 'artifacts/registry.jsonl'
if os.path.exists(reg):
    n = sum(1 for line in open(reg, encoding='utf-8') if line.strip())
    print('registry.jsonl restored from the clone: %d completed run(s)' % n)
else:
    print('no registry.jsonl in the clone -- first session, or it was never committed')


## 2. Install dependencies

Kaggle's base image already carries a CUDA build of torch. Let the pre-installed
wheel win rather than forcing a reinstall of the CPU pin from `requirements.txt`.


In [ ]:
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt

import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Settings -> Accelerator -> GPU T4 x2 (or P100).')


## 3. Point DATA_ROOT at the attached dataset

Dataset slug `raihanakirar/srp-dyna-card`, mounted at `/kaggle/input/srp-dyna-card`.
Note the `dataset/` wrapper directory inside the mount. Nothing is hardcoded: the
path comes from `configs/data.yaml` and is overridden here by the environment
variable, so the same code runs unchanged locally and on a Raspberry Pi.

`RESUME_FROM` below is the **fallback** resume path only. The primary path is the
committed `artifacts/registry.jsonl` that cell 1 already restored -- leave
`RESUME_FROM` at `None` unless that file is missing or stale because a previous
session's results never got committed.


In [ ]:
import os
os.environ['SRPCARD_DATA_ROOT'] = '/kaggle/input/srp-dyna-card/dataset'

# FALLBACK resume path -- normally leave this as None.
# The primary path is git: registry.jsonl is committed and arrived with the clone.
# Use this only when you could not commit the previous session's registry and
# uploaded artifacts/ as a private Kaggle Dataset instead. Files here OVERWRITE
# the committed ones, so a stale dataset will hide newer committed results.
RESUME_FROM = None   # e.g. '/kaggle/input/srp-card-artifacts'
if RESUME_FROM:
    import shutil, glob
    print('WARNING: using the RESUME_FROM fallback; these files overwrite the committed ones.')
    # Directories (artifacts/figures/) must be copied as trees. shutil.copy2 on a
    # directory raises, which used to abort the loop part-way and leave artifacts/
    # half restored -- a state that looks like a successful partial resume.
    for src in sorted(glob.glob(RESUME_FROM + '/*')):
        dst = os.path.join('artifacts', os.path.basename(src))
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print('restored:', sorted(os.listdir('artifacts')))

!ls -1 $SRPCARD_DATA_ROOT | head -20
!echo '--- images:' && find $SRPCARD_DATA_ROOT -type f | wc -l


## 4. Run ONE script

Uncomment exactly one line. Run them in this order across sessions:

| script | what | runs |
|---|---|---|
| `00_build_folds.py` | **preflight**, then index, dev split, folds. Verifies committed artefacts. | - |
| `01_complete_medium_grid.py` | 8 missing medium configs, legacy protocol | 8 |
| `02_lr_sweep_baselines.py` | baseline lr sweep | 6 |
| `03_run_cv.py` | the main experiment | 75 |
| `04_run_ablation.py` | class-weight ablation | 15 |
| `05_learning_curve.py` | learning curve, 5 fractions x 15 folds | 75 |
| `06_export_figures.py` | figures and tables | - |

**Run `00_build_folds.py` first in every fresh session.** Its phase 0 preflight
reports the GPU and torch/CUDA versions, whether `cudnn.deterministic` took effect,
the resolved DATA_ROOT with the per-class counts it actually found, whether the
committed artefacts still match their fingerprints, and -- the important one --
whether the pretrained checkpoint of all five arms downloads. A missing YOLO26
checkpoint shows up there, in the first minute, instead of 40 runs later. It exits
non-zero if any of that failed. `--preflight-only` runs just that part.

Scripts 01 and 02 write back into `configs/arms.yaml`. That file must be committed
before 03 runs, or 03 will use stale hyperparameters.

Every script is resumable: re-running skips what is already in the registry.

**On `--allow-pretrained-fallback`:** `configs/arms.yaml` names a YOLO11 checkpoint
as each YOLO arm's `pretrained_fallback`. It is never taken automatically -- a run
that trained YOLO11 while every table said YOLO26 would be undetectable afterwards.
If the YOLO26 download fails the script refuses and names both architectures. Pass
`--allow-pretrained-fallback` only if you have decided to accept the substitution;
it prints a loud banner and flags `pretrained_fallback_used` on every affected
registry record. Do not pass it by reflex to get a session unstuck.


In [ ]:
!python scripts/00_build_folds.py
# !python scripts/01_complete_medium_grid.py
# !python scripts/02_lr_sweep_baselines.py
# !python scripts/03_run_cv.py --quiet
# !python scripts/04_run_ablation.py --quiet
# !python scripts/05_learning_curve.py --quiet
# !python scripts/06_export_figures.py


## 5. Copy artifacts/ back out

**Run this cell even if the script above was interrupted.** The registry is
append-only and flushed after every run, so a partial `artifacts/` is still worth
keeping - it is exactly what lets the next session resume.

Everything under `/kaggle/working/artifacts_out/` is downloadable from the notebook
output panel. Then commit `artifacts/registry.jsonl` - that commit **is** the resume
mechanism - along with `configs/arms.yaml` if scripts 01 or 02 changed it, and any
new summary tables. See HANDOVER.md section 4 for the full list.


In [ ]:
import shutil, os, glob

# The bundle mirrors the REPOSITORY layout, not the contents of artifacts/.
# It used to hold artifacts/'s contents at its root plus arms.yaml beside them,
# while the docs said "unpack it over your local checkout" -- which would have
# put registry.jsonl at the repo root and left configs/arms.yaml untouched, so
# the resolved hyperparameters and the resume state were silently not committed.
OUT = '/kaggle/working/artifacts_out'
if os.path.exists(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT)

def stage(src, relpath):
    dst = os.path.join(OUT, relpath)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

for src in glob.glob('artifacts/**/*', recursive=True):
    # .bak files are local safety copies; they are gitignored and not results
    if os.path.isfile(src) and not src.endswith('.bak'):
        stage(src, os.path.relpath(src).replace(os.sep, '/'))

stage('configs/arms.yaml', 'configs/arms.yaml')

for root, _, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f)
        print('%9.1f KB  %s' % (os.path.getsize(p) / 1024, os.path.relpath(p, OUT)))

shutil.make_archive('/kaggle/working/artifacts_bundle', 'zip', OUT)
print()
print('bundle: /kaggle/working/artifacts_bundle.zip')
print('It unpacks straight over your local checkout: paths inside are')
print('artifacts/... and configs/arms.yaml, so nothing lands in the wrong place.')
print()
print('Then, in your local checkout:')
print('    unzip -o artifacts_bundle.zip')
print('    git add artifacts configs/arms.yaml && git status')
print('COMMIT artifacts/registry.jsonl -- that commit is the resume mechanism')
print('for the next session. See HANDOVER.md section 4.7.')


## 6. Progress check

How much is done, and how much is left.


In [ ]:
import json, collections
counts = collections.Counter()
fallback = collections.Counter()
corpora = collections.Counter()
unverified = 0
records = []
try:
    with open('artifacts/registry.jsonl', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))
except FileNotFoundError:
    print('no registry yet')

for r in records:
    counts[(r.get('script'), r.get('arm'))] += 1
    if r.get('pretrained_fallback_used'):
        fallback[(r.get('arm'), r.get('checkpoint_resolved'))] += 1
    if r.get('class_weights_verified') is False:
        unverified += 1
    fp = r.get('corpus_fingerprint') or {}
    corpora[(fp.get('kind'), fp.get('sha1_of_sorted_included_sha1s'))] += 1

for (script, arm), n in sorted(counts.items()):
    print('%-26s %-20s %3d' % (script, arm, n))
print('total records:', sum(counts.values()))

# Schema drift: a record missing a required field still matches by run_id, so its
# run is SKIPPED and its older numbers are inherited into the final results.
import sys
sys.path.insert(0, 'src')
from srpcard import registry
print()
registry.warn_if_stale()

print()
if fallback:
    print('*** %d record(s) used a PRETRAINED FALLBACK -- the architecture is not'
          ' what the arm declares:' % sum(fallback.values()))
    for (arm, ckpt), n in sorted(fallback.items()):
        print('      %-20s %-26s %3d run(s)' % (arm, ckpt, n))
    print('    These must not be reported as the declared architecture.')
else:
    print('no run used a pretrained fallback')

if unverified:
    print('*** %d record(s) have class_weights_verified=False' % unverified)
else:
    print('no run has a failed class-weight proof')

for (kind, sha1), n in sorted(corpora.items()):
    print('corpus %-16s %s  %3d run(s)' % (kind, sha1, n))
